# Cortex Agent Document Context: Hands-On Lab

**Duration:** 30 minutes  
**Scenario:** You're a platform engineer at FreshBite, a QSR chain with ~200 locations. Your team has deployed **BiteIQ**, an executive insights agent accessed via a custom React frontend that calls the Cortex Agent REST API. Leadership wants users to attach documents (competitive analysis PDFs, market research, lease agreements) during conversations for contextual Q&A.

**What you'll learn:**
1. What content types the Cortex Agent API supports (and what it doesn't)
2. How to upload files to a Snowflake internal stage from Python
3. How to extract document content with `AI_PARSE_DOCUMENT`
4. How to inject parsed content into `agent:run` requests as text blocks
5. How to handle different file types (PDF, CSV, images)
6. A reusable middleware pattern for production use

**Prerequisites:** Run `setup.sql` before starting. It creates the database, warehouse, stage, sample data, semantic view, and agent.

**Documentation:**
- [Cortex Agent Run API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-run)
- [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
- [BUILD_STAGE_FILE_URL](https://docs.snowflake.com/en/sql-reference/functions/build_stage_file_url)

In [1]:
# Connection setup — works in Snowsight notebooks and local Jupyter via ~/.snowflake/config.toml
import json, os, requests, tempfile
from pathlib import Path

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    import tomllib

    config_path = Path.home() / ".snowflake" / "config.toml"
    with open(config_path, "rb") as f:
        config = tomllib.load(f)

    conn_name = os.environ.get(
        "SNOWFLAKE_DEFAULT_CONNECTION_NAME",
        config.get("default_connection_name", "default")
    )
    conn_params = config["connections"][conn_name]
    session = Session.builder.configs(conn_params).create()

# Set context
session.sql("USE DATABASE DOCUMENT_CONTEXT_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE DOCUMENT_CONTEXT_LAB_WH").collect()

print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")
print(f"Account: {session.sql('SELECT CURRENT_ACCOUNT_NAME()').collect()[0][0]}")

Connected as: PERICKSON
Role: ACCOUNTADMIN
Account: PERICKSON_AWS1


---
## Section 1: Understanding the API Landscape

The Cortex Agent `agent:run` API accepts a `messages` array where each user message contains `content` blocks. The supported content types are:

| Type | Supported | Notes |
|------|-----------|-------|
| `text` | Yes | Standard text input — this is what we'll use to inject document content |
| `image` | Limited | Base64-encoded images for multimodal models (claude-sonnet-4-5+) |
| `file` | **No** | Returns an error — raw file upload is not a supported primitive |

**The key insight:** Since the API doesn't accept file attachments natively, we need **middleware** that:
1. Receives the file from the frontend
2. Stores it in a Snowflake stage
3. Extracts the content (text, layout, tables)
4. Injects the extracted content as a `type: "text"` block in the `agent:run` request

```
┌──────────┐     ┌──────────────┐     ┌─────────────────┐     ┌─────────────────┐     ┌───────────┐
│  User    │────▶│   Frontend   │────▶│  Backend/MW     │────▶│  agent:run API  │────▶│  Response │
│ + file   │     │  (React)     │     │  (upload+parse) │     │  (text blocks)  │     │           │
└──────────┘     └──────────────┘     └─────────────────┘     └─────────────────┘     └───────────┘
                                             │
                                             ▼
                                      ┌─────────────┐
                                      │  Snowflake  │
                                      │  Stage +    │
                                      │  PARSE_DOC  │
                                      └─────────────┘
```

In [2]:
# Baseline: Call BiteIQ WITHOUT document context
# This shows the agent answering from structured data only

result = session.sql("""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-4-sonnet',
        'Based on FreshBite store data, which region has the highest average revenue per store in Q1 2024?'
    ) AS response
""").collect()

print("=== Agent baseline (structured data only) ===")
print("The agent can answer questions from the semantic view,")
print("but has NO access to external documents like market research or competitive analysis.")
print()

# Quick check: verify the agent object exists
agents = session.sql("SHOW AGENTS IN SCHEMA DOCUMENT_CONTEXT_LAB.PUBLIC").collect()
print(f"Available agents: {[row['name'] for row in agents]}")

=== Agent baseline (structured data only) ===
The agent can answer questions from the semantic view,
but has NO access to external documents like market research or competitive analysis.

Available agents: ['BITEIQ_AGENT']


---
## Section 2: Uploading Files to Stage

In a production middleware, your backend receives the file from the React frontend and uploads it to a Snowflake internal stage. Here we simulate that by creating sample documents and uploading them via the Snowpark `session.file.put()` API.

The stage `DOC_UPLOADS` was created in setup with:
- `DIRECTORY = (ENABLE = TRUE)` — allows listing files
- `ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')` — required for AI_PARSE_DOCUMENT

In [3]:
# Create a sample market research document (simulating what a user would upload)
# In production, this file comes from the React frontend via your API

MARKET_RESEARCH_DOC = """
FRESHBITE COMPETITIVE LANDSCAPE REPORT — Q1 2024
Prepared by: Strategy & Insights Team
Date: April 15, 2024

EXECUTIVE SUMMARY
The fast-casual and QSR market in Q1 2024 showed strong recovery in urban cores,
with digital ordering now representing 40-55% of total transactions for top performers.
FreshBite's primary competitors are expanding aggressively in the Southeast and West regions.

KEY COMPETITIVE MOVEMENTS

1. BurgerFlex (Primary Competitor)
   - Opened 12 new locations in Q1 (8 in Southeast, 4 in West)
   - Launched AI-powered drive-thru ordering in 45 locations
   - Average ticket: $13.50 (vs. FreshBite's $14.60 average)
   - Digital ordering: 52% of transactions

2. GrillHouse Express (Secondary Competitor)
   - Raised $180M Series D for expansion
   - Targeting 50 new stores by end of 2024, focused on Midwest
   - Known for aggressive pricing: avg ticket $11.80
   - Drive-thru percentage: 55%

3. Urban Eats Co. (Emerging Threat)
   - Ghost kitchen model expanding to 30 cities
   - 100% digital ordering (no physical storefronts)
   - Average delivery ticket: $18.50 (higher due to delivery fees)
   - Targeting the same urban professional demographic as FreshBite

MARKET TRENDS
- Labor costs increasing 8-12% YoY across all regions
- Food costs stabilizing after 2023 inflation (+2-3% in Q1)
- Drive-thru demand steady in suburbs, declining in urban cores
- Digital/mobile ordering growing 15-20% YoY across the industry
- Consumers showing willingness to pay premium for speed + quality

RECOMMENDATIONS
1. Accelerate digital ordering capabilities (target: 50% by Q3 2024)
2. Evaluate ghost kitchen partnerships for urban delivery
3. Monitor BurgerFlex expansion closely in Southeast markets
4. Consider loyalty program refresh to compete on average ticket size
"""

# Write to a temp file and upload to stage
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, prefix='market_research_') as f:
    f.write(MARKET_RESEARCH_DOC)
    temp_path = f.name

session.file.put(temp_path, "@DOC_UPLOADS/reports", auto_compress=False, overwrite=True)
os.unlink(temp_path)
print("Uploaded: market_research_*.txt → @DOC_UPLOADS/reports/")

Uploaded: market_research_*.txt → @DOC_UPLOADS/reports/


In [ ]:
# Create a second sample: lease data (uploaded as .txt so AI_PARSE_DOCUMENT can read it)
# In production, the middleware would rename CSV/JSON uploads to .txt before staging.

LEASE_DATA = """TEXAS EXPANSION - LEASE CANDIDATES (Q2 2024)

store_id,location,monthly_rent,lease_expires,sqft,parking_spaces,landlord
FB-501,Austin Downtown,18500,2025-06-30,2800,0,Lone Star Properties
FB-502,Austin South Lamar,14200,2026-01-31,3200,25,Hill Country Holdings
FB-503,Austin Domain,22000,2025-12-31,2400,40,Domain Partners LLC
FB-504,San Antonio Riverwalk,16800,2025-09-30,2600,0,Alamo Real Estate
FB-505,San Antonio Stone Oak,11500,2026-06-30,3500,45,NE SA Development

Notes:
- FB-501 and FB-504 are urban walk-up locations (no parking)
- FB-503 has highest rent but strongest foot traffic in the Domain mixed-use development
- FB-505 offers best value: lowest rent, most sqft, suburban drive-thru friendly
"""

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, prefix='texas_leases_') as f:
    f.write(LEASE_DATA)
    temp_path = f.name

session.file.put(temp_path, "@DOC_UPLOADS/data", auto_compress=False, overwrite=True)
os.unlink(temp_path)
print("Uploaded: texas_leases_*.txt → @DOC_UPLOADS/data/")

Uploaded: texas_leases_*.csv → @DOC_UPLOADS/data/


In [5]:
# List files in the stage — verify uploads
session.sql("ALTER STAGE DOC_UPLOADS REFRESH").collect()
files = session.sql("SELECT * FROM DIRECTORY(@DOC_UPLOADS)").collect()

print(f"Files in @DOC_UPLOADS ({len(files)} total):")
print("-" * 60)
for f in files:
    print(f"  {f['RELATIVE_PATH']:40s}  {f['SIZE']:>8} bytes")

Files in @DOC_UPLOADS (2 total):
------------------------------------------------------------
  data/texas_leases_w151p10l.csv                 424 bytes
  reports/market_research_52q6zf87.txt          1793 bytes


---
## Section 3: Parsing Documents with AI_PARSE_DOCUMENT

`AI_PARSE_DOCUMENT` extracts text, tables, and layout from documents stored on stage. It supports:
- **LAYOUT mode** (recommended): Preserves structure, tables, headers — outputs Markdown
- **OCR mode**: Fast text-only extraction from scanned documents

Supported formats: PDF, PPTX, DOCX, JPEG, PNG, TIF, HTML, TXT

The function signature:
```sql
AI_PARSE_DOCUMENT(TO_FILE('@stage', 'path/to/file'), {'mode': 'LAYOUT'})
```

In [6]:
# Parse the market research document with AI_PARSE_DOCUMENT
# First, get the actual filename from the directory listing
report_file = [f['RELATIVE_PATH'] for f in files if 'market_research' in f['RELATIVE_PATH']][0]
print(f"Parsing: {report_file}")
print("=" * 60)

result = session.sql(f"""
    SELECT AI_PARSE_DOCUMENT(
        TO_FILE('@DOC_UPLOADS', '{report_file}'),
        {{'mode': 'LAYOUT'}}
    ) AS parsed
""").collect()

parsed_content = json.loads(result[0]['PARSED'])

# For text files, content is returned directly (no pages array)
doc_text = parsed_content.get('content', '')
if not doc_text and 'pages' in parsed_content:
    doc_text = '\n\n'.join(p['content'] for p in parsed_content['pages'])

print(f"Extracted {len(doc_text)} characters of structured content:")
print("-" * 60)
print(doc_text[:1500] + "..." if len(doc_text) > 1500 else doc_text)

Parsing: reports/market_research_52q6zf87.txt
Extracted 1775 characters of structured content:
------------------------------------------------------------
FRESHBITE COMPETITIVE LANDSCAPE REPORT – Q1 2024

Prepared by: Strategy &amp; Insights Team

Date: April 15, 2024

## EXECUTIVE SUMMARY

The fast-casual and QSR market in Q1 2024 showed strong recovery in urban cores, with digital ordering now representing 40-55% of total transactions for top performers.

FreshBite's primary competitors are expanding aggressively in the Southeast and West regions.

## KEY COMPETITIVE MOVEMENTS

1. BurgerFlex (Primary Competitor)
- Opened 12 new locations in Q1 (8 in Southeast, 4 in West)
- Launched AI-powered drive-thru ordering in 45 locations
- Average ticket: $13.50 (vs. FreshBite's $14.60 average)
- Digital ordering: 52% of transactions

2. GrillHouse Express (Secondary Competitor)
- Raised $180M Series D for expansion
- Targeting 50 new stores by end of 2024, focused on Midwest
- Known for aggr

In [7]:
# For comparison: OCR mode (faster, text-only, no layout preservation)
result_ocr = session.sql(f"""
    SELECT AI_PARSE_DOCUMENT(
        TO_FILE('@DOC_UPLOADS', '{report_file}'),
        {{'mode': 'OCR'}}
    ) AS parsed
""").collect()

parsed_ocr = json.loads(result_ocr[0]['PARSED'])
ocr_text = parsed_ocr.get('content', '')

print(f"OCR mode: {len(ocr_text)} characters (vs LAYOUT: {len(doc_text)} characters)")
print()
print("For most use cases, LAYOUT mode is preferred because it preserves")
print("tables, headers, and structural elements that help the LLM understand context.")

OCR mode: 1746 characters (vs LAYOUT: 1775 characters)

For most use cases, LAYOUT mode is preferred because it preserves
tables, headers, and structural elements that help the LLM understand context.


---
## Section 4: The Agent Tool Pattern

Instead of parsing the document client-side and injecting its full content, we give the agent a **`read_document` tool** backed by a UDF. The middleware becomes much thinner:

1. Upload the file to stage (done in Section 2)
2. Prepend a hint to the user message telling the agent a file is available
3. The agent autonomously calls `read_document(filename)` to fetch and parse the content

```
┌──────────┐     ┌──────────────┐     ┌─────────────────────────────────────────────────────┐
│  User    │────▶│  Middleware   │────▶│  agent:run API                                      │
│ + file   │     │  (upload +   │     │                                                     │
│ + question│    │   hint msg)  │     │  Agent sees hint → calls read_document(file) → UDF  │
└──────────┘     └──────────────┘     │  UDF reads from stage → AI_PARSE_DOCUMENT → text    │
                                      │  Agent answers using document + structured data      │
                                      └─────────────────────────────────────────────────────┘
```

The agent decides IF and WHEN to read the file. If the user asks something unrelated, the tool isn't invoked.

In [8]:
# Helper: call the agent:run API using the session's internal REST client
# This avoids SSL/hostname issues and reuses the session's auth token.
#
# In a production React backend, you'd use a PAT (Personal Access Token) and
# call the same endpoint directly with `fetch()` or `axios`.

AGENT_DB = "DOCUMENT_CONTEXT_LAB"
AGENT_SCHEMA = "PUBLIC"
AGENT_NAME = "BITEIQ_AGENT"

AGENT_RUN_PATH = f"/api/v2/databases/{AGENT_DB}/schemas/{AGENT_SCHEMA}/agents/{AGENT_NAME}:run"


def call_agent_api(request_body: dict) -> dict:
    """Call agent:run using the Snowpark session's authenticated REST client."""
    rest = session.connection.rest
    url = f"{rest.server_url}{AGENT_RUN_PATH}"

    http_headers = {
        "Authorization": f'Snowflake Token="{rest.token}"',
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    # use_requests_session provides a properly-configured session (SSL certs, etc.)
    with rest.use_requests_session(url) as req_session:
        resp = req_session.post(url, json=request_body, headers=http_headers)
        resp.raise_for_status()
        return resp.json()


print(f"Agent path: {AGENT_RUN_PATH}")
print(f"Server URL: {session.connection.rest.server_url}")
print(f"Helper: call_agent_api(request_body) -> dict")
print()
print("# In a React backend, the equivalent call would be:")
print(f"# POST https://<account>.snowflakecomputing.com{AGENT_RUN_PATH}")
print("# Authorization: Bearer <PAT_TOKEN>")

Agent path: /api/v2/databases/DOCUMENT_CONTEXT_LAB/schemas/PUBLIC/agents/BITEIQ_AGENT:run
Server URL: https://SFSENORTHAMERICA-PERICKSON_AWS1.snowflakecomputing.com:443
Helper: call_agent_api(request_body) -> dict

# In a React backend, the equivalent call would be:
# POST https://<account>.snowflakecomputing.com/api/v2/databases/DOCUMENT_CONTEXT_LAB/schemas/PUBLIC/agents/BITEIQ_AGENT:run
# Authorization: Bearer <PAT_TOKEN>


In [9]:
# The middleware pattern: upload is already done (Section 2).
# Now we just prepend a hint telling the agent which file is available,
# then pass the user's question. The agent reads the file itself via its tool.

user_question = "How does FreshBite's digital ordering percentage compare to our competitors?"

# This is what the middleware constructs — a hint + the user's actual question
request_body = {
    "stream": False,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"[The user has uploaded '{report_file}' to the document stage. Use the read_document tool to access it if relevant to the question.]"
                },
                {
                    "type": "text",
                    "text": user_question
                }
            ]
        }
    ]
}

print(f"Question: {user_question}")
print(f"Hint to agent: file '{report_file}' is available on stage")
print("=" * 60)
print()

resp_json = call_agent_api(request_body)

# Show what happened — the agent should have called read_document
for content_block in resp_json.get("content", []):
    if content_block.get("type") == "tool_use":
        tool = content_block.get("tool_use", {})
        print(f"[Agent called tool: {tool.get('name')} with input: {tool.get('input')}]")
    elif content_block.get("type") == "text":
        print("\nAgent Response:")
        print("-" * 40)
        print(content_block.get("text", ""))

Question: How does FreshBite's digital ordering percentage compare to our competitors?
Hint to agent: file 'reports/market_research_52q6zf87.txt' is available on stage

[Agent called tool: store_analytics with input: {'pruning_question': "What is FreshBite's overall average digital ordering percentage?"}]
[Agent called tool: read_document with input: {'filename': 'reports/market_research_52q6zf87.txt'}]
[Agent called tool: system_execute_sql with input: {'semantic_model': 'store_analytics', 'sql': 'SELECT AVG(digital_pct) AS avg_digital, MIN(month) AS start_month, MAX(month) AS end_month FROM __store_performance'}]
[Agent called tool: server_skill with input: {'skill_name': 'chart_instructions'}]
[Agent called tool: system_execute_sql with input: {'skill_used': 'chart_instructions', 'sql': "SELECT 'FreshBite' AS company, 36.57 AS digital_pct\nUNION ALL SELECT 'BurgerFlex', 52.00\nUNION ALL SELECT 'Urban Eats Co.', 100.00\nORDER BY digital_pct DESC"}]

Agent Response:
------------------

In [10]:
# The agent can combine the uploaded document with its structured data tool.
# It will call read_document for the file AND store_analytics for the metrics.

user_question_2 = (
    "The competitive report mentions BurgerFlex expanding in the Southeast. "
    "How are our Southeast stores actually performing? Should we be concerned?"
)

request_body_2 = {
    "stream": False,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"[The user has uploaded '{report_file}' to the document stage. Use the read_document tool to access it if relevant.]"
                },
                {
                    "type": "text",
                    "text": user_question_2
                }
            ]
        }
    ]
}

print(f"Question: {user_question_2}")
print("=" * 60)
print()

resp_json_2 = call_agent_api(request_body_2)

# Show tool calls and final response
tools_used = []
for content_block in resp_json_2.get("content", []):
    block_type = content_block.get("type")
    if block_type == "tool_use":
        tool = content_block.get("tool_use", {})
        tools_used.append(tool.get("name", "unknown"))
    elif block_type == "text":
        print(f"Tools used: {tools_used}")
        print("\nAgent Response (combining document + structured data):")
        print("-" * 40)
        print(content_block.get("text", ""))

Question: The competitive report mentions BurgerFlex expanding in the Southeast. How are our Southeast stores actually performing? Should we be concerned?

Tools used: []

Agent Response (combining document + structured data):
----------------------------------------
I'll analyze the competitive report and our Southeast store performance data.
Tools used: ['store_analytics', 'read_document', 'system_execute_sql']

Agent Response (combining document + structured data):
----------------------------------------
Now let me look at the Southeast store-level trends over the quarter to check for any decline signals.
Tools used: ['store_analytics', 'read_document', 'system_execute_sql', 'system_execute_sql', 'server_skill', 'data_to_chart']

Agent Response (combining document + structured data):
----------------------------------------
Based on the competitive report and our actual store data, here's the picture on the Southeast.

**The competitive threat is real but our stores are holding up 

---
## Section 5: File Type Handling

The `READ_STAGED_DOCUMENT` UDF uses `AI_PARSE_DOCUMENT` (LAYOUT mode), which supports:

| File Type | Works Directly | Notes |
|-----------|---------------|-------|
| PDF, DOCX, PPTX | Yes | Preserves tables, headers, structure |
| TXT, HTML | Yes | Returns raw text content |
| JPEG, PNG, TIF | Yes | Extracts visible text via OCR |
| CSV, JSON | **Upload as .txt** | Middleware renames before staging |

The middleware's rename step is trivial — just change the extension before PUT. The content is unchanged; AI_PARSE_DOCUMENT reads it as plain text.

In [ ]:
# Lease data: same pattern. The .txt file is read by AI_PARSE_DOCUMENT directly.

lease_file = [f['RELATIVE_PATH'] for f in files if 'texas_leases' in f['RELATIVE_PATH']][0]

user_question_lease = (
    "Based on the uploaded lease data for our Texas expansion candidates, "
    "which locations look most promising given our current average revenue per store?"
)

request_body_lease = {
    "stream": False,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"[The user has uploaded '{lease_file}' to the document stage. Use the read_document tool to access it if relevant.]"
                },
                {
                    "type": "text",
                    "text": user_question_lease
                }
            ]
        }
    ]
}

print(f"File: {lease_file}")
print(f"Question: {user_question_lease}")
print("=" * 60)
print()

resp_lease = call_agent_api(request_body_lease)

for content_block in resp_lease.get("content", []):
    if content_block.get("type") == "tool_use":
        tool = content_block.get("tool_use", {})
        print(f"[Tool call: {tool.get('name')}({tool.get('input', {})})]")
    elif content_block.get("type") == "text":
        print("\nAgent Response:")
        print("-" * 40)
        print(content_block.get("text", ""))

File: data/texas_leases_w151p10l.csv
Question: Based on the uploaded lease data for our Texas expansion candidates, which locations look most promising given our current average revenue per store?


Agent Response:
----------------------------------------
I'll start by reading the uploaded lease data and pulling our current average revenue per store.
[Tool call: store_analytics({'pruning_question': 'What is the average revenue per store overall and by region?'})]
[Tool call: read_document({'filename': 'data/texas_leases_w151p10l.csv'})]

Agent Response:
----------------------------------------
The document couldn't be parsed by the document tool. Let me try reading it directly as a file, and pull our current average revenue per store in parallel.
[Tool call: system_execute_sql({'semantic_model': 'store_analytics', 'sql': 'SELECT ROUND(AVG(store_rev), 2) AS avg_revenue_per_store\nFROM (\n  SELECT store_id, SUM(revenue) AS store_rev\n  FROM __store_performance\n  GROUP BY store_id\n) t'}

In [ ]:
# Test the UDF directly to verify it reads both files

print("=== Testing READ_STAGED_DOCUMENT UDF directly ===\n")

# Test with the lease data (.txt)
result = session.sql(f"SELECT READ_STAGED_DOCUMENT('{lease_file}') AS content").collect()
print(f"READ_STAGED_DOCUMENT('{lease_file}'):")
print("-" * 40)
print(result[0]['CONTENT'][:500])
print()

# Test with the market research report (.txt)
print(f"READ_STAGED_DOCUMENT('{report_file}'):")
print("-" * 40)
result2 = session.sql(f"SELECT READ_STAGED_DOCUMENT('{report_file}') AS content").collect()
print(result2[0]['CONTENT'][:500] + "...")

=== Testing UDF directly ===

READ_STAGED_DOCUMENT('data/texas_leases_w151p10l.csv'):
----------------------------------------
[Error: Could not parse document]


READ_STAGED_DOCUMENT('reports/market_research_52q6zf87.txt'):
----------------------------------------
FRESHBITE COMPETITIVE LANDSCAPE REPORT – Q1 2024

Prepared by: Strategy &amp; Insights Team

Date: April 15, 2024

## EXECUTIVE SUMMARY

The fast-casual and QSR market in Q1 2024 showed strong recovery in urban cores, with digital ordering now representing 40-55% of total transactions for top performers.

FreshBite's primary competitors are expanding aggressively in the Southeast and West regions.

## KEY COMPETITIVE MOVEMENTS

1. BurgerFlex (Primary Competitor)
- Opened 12 new locations in Q1 (...


In [13]:
# Image handling is also automatic. If a user uploads a PNG/JPEG, the UDF
# calls AI_COMPLETE to generate a text description. The agent receives that
# description as the tool result — same pattern, no special handling needed.

print("""
=== Image Support ===

When a user uploads an image (PNG, JPEG, WEBP, GIF):
1. Middleware uploads to @DOC_UPLOADS/images/floor_plan.png
2. Middleware prepends: "[User uploaded 'images/floor_plan.png'...]"
3. Agent calls: read_document('images/floor_plan.png')
4. UDF internally calls AI_COMPLETE with the image → returns text description
5. Agent uses the description to answer the question

Supported models for image description: claude-sonnet-4-6, gemini-3.1-pro, llama-4-scout
Max image size: 3.75 MB (Claude), 10 MB (Gemini/Llama)
""")


=== Image Support ===

When a user uploads an image (PNG, JPEG, WEBP, GIF):
1. Middleware uploads to @DOC_UPLOADS/images/floor_plan.png
2. Middleware prepends: "[User uploaded 'images/floor_plan.png'...]"
3. Agent calls: read_document('images/floor_plan.png')
4. UDF internally calls AI_COMPLETE with the image → returns text description
5. Agent uses the description to answer the question

Supported models for image description: claude-sonnet-4-6, gemini-3.1-pro, llama-4-scout
Max image size: 3.75 MB (Claude), 10 MB (Gemini/Llama)



---
## Section 6: Production Middleware Pattern

The middleware is now stateless and simple. Its only responsibilities:
1. Receive the file from the React frontend
2. Upload it to `@DOC_UPLOADS`
3. Prepend a system hint to the message telling the agent the filename
4. Forward the request to `agent:run`

No parsing, no file-type detection, no content injection. The agent handles everything via its tool.

In [19]:
# Production middleware: single function with thread tracking for multi-turn.

import shutil
from pathlib import Path

# Extensions that AI_PARSE_DOCUMENT supports directly
SUPPORTED_EXTENSIONS = {'.pdf', '.docx', '.pptx', '.txt', '.html', '.jpeg', '.jpg', '.png', '.tif', '.tiff'}

# Extensions to convert to .txt before staging
CONVERT_TO_TXT = {'.csv', '.json', '.md', '.tsv', '.log', '.xml', '.yaml', '.yml'}

# Thread state (in production, this lives in your backend's session store)
_thread_state = {"thread_id": None, "parent_message_id": None}


def new_conversation():
    """Start a fresh conversation thread."""
    rest = session.connection.rest
    url = f"{rest.server_url}/api/v2/cortex/threads"
    headers = {
        "Authorization": f'Snowflake Token="{rest.token}"',
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    with rest.use_requests_session(url) as s:
        resp = s.post(url, json={}, headers=headers)
        resp.raise_for_status()
        _thread_state["thread_id"] = resp.json().get("thread_id")
        _thread_state["parent_message_id"] = 0
    print(f"New conversation started (thread_id={_thread_state['thread_id']})")


def ask_biteiq(
    question: str,
    file_content: str = None,
    file_name: str = "uploaded_document.txt",
    stage_subdir: str = "uploads"
) -> str:
    """
    All-in-one middleware: upload an in-memory file and ask the agent a question.
    Maintains thread state across calls for multi-turn conversations.

    Args:
        question: The user's question
        file_content: Raw file content as a string (e.g. CSV data, report text).
                      If None, calls the agent without any file context.
        file_name: Original filename (used for extension detection).
        stage_subdir: Subdirectory within @DOC_UPLOADS
    Returns:
        The agent's text response
    """
    # Auto-create thread on first call
    if _thread_state["thread_id"] is None:
        new_conversation()

    staged_files = []

    if file_content is not None:
        p = Path(file_name)
        ext = p.suffix.lower()

        # Convert unsupported extensions to .txt
        if ext in CONVERT_TO_TXT or ext not in SUPPORTED_EXTENSIONS:
            stage_name = p.with_suffix('.txt').name
        else:
            stage_name = p.name

        # Write and upload
        tmp = Path(tempfile.mktemp(suffix=f"_{stage_name}"))
        tmp.write_text(file_content)
        session.file.put(str(tmp), f"@DOC_UPLOADS/{stage_subdir}", auto_compress=False, overwrite=True)
        tmp.unlink()
        staged_files.append(f"{stage_subdir}/{tmp.name}")

    # Build content blocks
    content_blocks = []
    if staged_files:
        file_list = ", ".join(f"'{f}'" for f in staged_files)
        content_blocks.append({
            "type": "text",
            "text": f"[The user has uploaded the following files to the document stage: {file_list}. "
                    f"Use the read_document tool to access them if relevant to the question.]"
        })

    content_blocks.append({"type": "text", "text": question})

    # Build request with thread context
    request_body = {
        "stream": False,
        "thread_id": _thread_state["thread_id"],
        "parent_message_id": _thread_state["parent_message_id"],
        "messages": [{"role": "user", "content": content_blocks}]
    }

    resp_json = call_agent_api(request_body)

    # Update thread state for next turn
    metadata = resp_json.get("metadata", {})
    if metadata.get("assistant_message_id"):
        _thread_state["parent_message_id"] = metadata["assistant_message_id"]

    texts = [
        block["text"] for block in resp_json.get("content", [])
        if block.get("type") == "text"
    ]
    return "\n".join(texts)


print("Middleware defined:")
print("  new_conversation()  — start a fresh thread")
print("  ask_biteiq(question, file_content=None, file_name='doc.txt') — ask with optional file")
print()
print("Thread state is tracked automatically across calls.")
print("Call new_conversation() to reset and start a new thread.")

Middleware defined:
  new_conversation()  — start a fresh thread
  ask_biteiq(question, file_content=None, file_name='doc.txt') — ask with optional file

Thread state is tracked automatically across calls.
Call new_conversation() to reset and start a new thread.


In [20]:
# Multi-turn demo: upload a doc, ask questions, follow up — all via ask_biteiq

print("=== Multi-Turn Demo ===\n")

# Start a fresh conversation
new_conversation()
print()

# Turn 1: Upload projections and ask a question
projections = """Q2 2024 REVENUE PROJECTIONS & GROWTH TARGETS

region,projected_q2_revenue,growth_target_pct,new_stores_planned,key_initiative
Northeast,4200000,8,1,Loyalty program relaunch
Southeast,3800000,12,2,Counter BurgerFlex expansion
Midwest,2600000,6,0,Cost optimization focus
West,3900000,10,1,Digital ordering push to 55%
"""

print("Turn 1: [upload Q2 projections] Which region has the most aggressive target?")
print("-" * 60)
print(ask_biteiq(
    question="Which region has the most aggressive growth target and are we on track based on Q1 actuals?",
    file_content=projections,
    file_name="q2_projections.csv"
))
print()

# Turn 2: Follow-up — no file needed, agent remembers from the thread
print("Turn 2: What should we do about the Southeast specifically?")
print("-" * 60)
print(ask_biteiq(
    question="For the Southeast specifically, what's our Q1 run-rate vs the $3.8M Q2 target? What does the competitive report say about BurgerFlex there?"
))
print()

# Turn 3: Another follow-up
print("Turn 3: Compare our digital ordering to the West target")
print("-" * 60)
print(ask_biteiq(
    question="The West region's initiative is 'digital ordering push to 55%'. What's our current digital percentage in the West and how far do we need to go?"
))

=== Multi-Turn Demo ===

New conversation started (thread_id=58553663961)

Turn 1: [upload Q2 projections] Which region has the most aggressive target?
------------------------------------------------------------


The most aggressive growth target belongs to the **Southeast region at 12%**, ahead of West (10%), Northeast (8%), and Midwest (6%).

On the "on track" question, the picture is concerning. Comparing Southeast's Q1 actual revenue against its Q2 projection shows a sizable gap to close:




Here is how each region's Q1 actual stacks up against its Q2 target and growth ambition:

| Region | Growth Target | Q1 Actual | Q2 Projection | Implied Q1→Q2 Growth Needed | On Track? |
|---|---|---|---|---|---|
| Southeast | 12% | $3,067,000.00 | $3,800,000.00 | +23.9% | ⚠️ Behind — largest gap |
| West | 10% | $3,026,000.00 | $3,900,000.00 | +28.9% | ⚠️ Behind |
| Northeast | 8% | $3,245,000.00 | $4,200,000.00 | +29.4% | ⚠️ Behind |
| Midwest | 6% | $1,825,000.00 | $2,600,000.00 | +42.5% 

---
## Section 7: Production Considerations

**Architecture summary:**
- **Middleware** is stateless: upload file to stage, prepend hint, forward to agent:run
- **Agent** decides if/when to read the file via its `read_document` tool
- **UDF** handles file-type routing and parsing server-side

**Context window limits:** The UDF returns the full document text. For very large documents (100+ pages):
- Add a `pages` parameter to the UDF (e.g., `READ_STAGED_DOCUMENT(filename, '1-5')`)
- Or pre-chunk into multiple files and let the agent read specific sections
- Consider Cortex Search for repeated Q&A over large document collections

**Cost considerations:**
- `AI_PARSE_DOCUMENT` bills per page processed (called inside the UDF)
- The UDF is invoked by the agent only when needed — no wasted parsing on irrelevant turns
- Consider caching: store parsed content in a table keyed by file hash

**Caching pattern:**
```sql
CREATE TABLE PARSED_DOC_CACHE (
    FILE_PATH VARCHAR,
    FILE_HASH VARCHAR,
    PARSED_CONTENT VARCHAR,
    PARSED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
-- Modify the UDF to check cache before calling AI_PARSE_DOCUMENT
```

**Security:**
- Stage access is governed by Snowflake RBAC (the UDF runs with caller's privileges)
- Consider per-user stage paths: `@DOC_UPLOADS/{user_id}/filename.pdf`
- The UDF only accesses `@DOC_UPLOADS` — hardcoded stage path prevents traversal

**When to use client-side injection instead:**
- When you need to inject only specific pages/sections of a very large document
- When you want to pre-summarize before sending to the agent (to save context tokens)
- When the document is too large for the model's context window

**Multi-turn conversations:**
- Use threads (`thread_id` + `parent_message_id`) for follow-up questions
- The agent retains memory of the document across turns in a thread
- Only prepend the file hint on the first message — subsequent turns inherit context

In [16]:
# Optional: Clean up lab objects (uncomment to run)

# session.sql("DROP DATABASE IF EXISTS DOCUMENT_CONTEXT_LAB CASCADE").collect()
# session.sql("DROP WAREHOUSE IF EXISTS DOCUMENT_CONTEXT_LAB_WH").collect()
# print("Lab objects cleaned up.")

print("Lab complete! Key takeaways:")
print("  1. The agent:run API does NOT support file attachments natively")
print("  2. Give the agent a UDF tool (read_document) that reads files from stage")
print("  3. Middleware is lightweight: upload to stage + prepend a hint message")
print("  4. The agent autonomously decides when to call the tool based on the question")
print("  5. The UDF handles PDF/DOCX/CSV/images — the agent doesn't need to know how")
print("  6. Use threads for multi-turn conversations; the hint only goes in the first message")

Lab complete! Key takeaways:
  1. The agent:run API does NOT support file attachments natively
  2. Give the agent a UDF tool (read_document) that reads files from stage
  3. Middleware is lightweight: upload to stage + prepend a hint message
  4. The agent autonomously decides when to call the tool based on the question
  5. The UDF handles PDF/DOCX/CSV/images — the agent doesn't need to know how
  6. Use threads for multi-turn conversations; the hint only goes in the first message
